# Analisis de la Red de Transferencias de Tokens Ethereum
## Proyecto Final - Estructura de Datos

**Profesor:** Jonnathan Alfredo Ramos Chaux  
**Equipo:** Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TU_USUARIO/TU_REPO/blob/main/notebooks/Proyecto_Teoria_Grafos.ipynb)

---

### División de tareas y algoritmos

| Sección | Descripción | Responsable | Algoritmo principal |
|---------|-------------|-------------|---------------------|
| Exploración | Análisis previo del dataset | Pablo Álvarez | — |
| Tarea 1 | Caminos Mínimos | Pablo Álvarez | Bellman-Ford |
| Tarea 2 | Centralidad e Influencia | Jhon Arciniegas | PageRank y Betweenness |
| Tarea 3 | Detección de Comunidades | Emerson Ballesteros | Louvain |
| Tarea 4 | Árbol de Expansión Mínima | Trabajo grupal | Kruskal (Union-Find) |

**Nota importante:** De acuerdo con los lineamientos del proyecto, el uso del algoritmo de Dijkstra está estrictamente prohibido en todas las implementaciones.

### Configuración del Entorno (Google Colab)

In [6]:
# Importación de librerías estándar
import csv
import os
import random
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Verificación de NetworkX (Uso exclusivo para visualización, no para lógica de grafos)
try:
    import networkx as nx
    NX_AVAILABLE = True
    print("NetworkX cargado (restringido a visualizaciones auxiliares).")
except ImportError:
    NX_AVAILABLE = False
    print("Aviso: NetworkX no está instalado. Se omitirán las visualizaciones avanzadas.")

# Importación de los módulos del proyecto
from src.graph_loader import EthereumGraphLoader
from src.bellman_ford import bellman_ford, reconstruct_path
from src.centrality import degree_centrality, pagerank, betweenness_centrality_approx, top_k_nodes
from src.community_detection import symmetrize_graph, louvain_communities, compute_modularity, community_summary
from src.spanning_tree import kruskal_mst, mst_analytics, symmetrize_edges, UnionFind

print("Módulos locales importados correctamente.")

Aviso: NetworkX no está instalado. Se omitirán las visualizaciones avanzadas.


ModuleNotFoundError: No module named 'src'

---

### Sección 1: Exploración del Dataset (10%)
**Responsable:** Pablo Álvarez

#### 1.1 Descripción de los datos

El dataset está compuesto por un registro histórico de transferencias de tokens ERC-20 en la red de Ethereum, construido a partir de los eventos `Transfer(address indexed from, address indexed to, uint256 value)` emitidos por los contratos inteligentes.

**Archivos principales:**

| Archivo | Tamaño aprox. (Filas) | Descripción |
|---------|-----------------------|-------------|
| `token_transfers_full.csv` | ~38.9M | Historial central de transferencias ERC-20 |
| `exchangeLabels.csv` | 296 | Direcciones etiquetadas de exchanges y DEX |
| `alphacore_labels_stablecoin.csv` | 181 | Direcciones de stablecoins y pools de liquidez |

**Estructura de `token_transfers_full.csv`:**

| Columna | Tipo de dato | Descripción |
|---------|--------------|-------------|
| `token_address` | String (Hex) | Dirección del contrato inteligente del token |
| `from_address` | String (Hex) | Dirección de la billetera origen (emisor) |
| `to_address` | String (Hex) | Dirección de la billetera destino (receptor) |
| `value` | Entero (uint256)| Monto transferido (en la unidad mínima del token) |
| `transaction_hash` | String (Hex) | Identificador único de la transacción en la red |
| `log_index` | Entero | Índice de posición del log dentro del bloque |
| `block_number` | Entero | Número del bloque donde se validó la transacción |

#### 1.2 Modelado del Grafo

Para los propósitos de este proyecto, la red de transacciones se modela computacionalmente como un grafo dirigido ponderado:

$$G = (V, E, w)$$

Donde:
* **$V$ (Vértices):** Conjunto de direcciones únicas (billeteras), derivado de la unión entre `from_address` y `to_address`.
* **$E$ (Aristas):** Pares ordenados $(u, v)$ que indican que existió al menos una transferencia desde la dirección $u$ hacia la dirección $v$.
* **$w(e)$ (Pesos):** Volumen total transferido, calculado como la suma de todos los valores de las transacciones históricas entre ese par de nodos.

In [ ]:
# Carga de datos y construcción del grafo
# Se utiliza una muestra reducida para agilizar el desarrollo y las pruebas iniciales.

# Rutas relativas de los archivos (ajustar si se monta desde Google Drive)
SAMPLE_PATH   = 'data/samples/token_transfers_sample.csv'
FULL_PATH     = 'data/raw/token_transfers_full.csv'
EXCHANGE_PATH = 'data/raw/exchangeLabels.csv'
STABLE_PATH   = 'data/raw/alphacore_labels_stablecoin.csv'

# Bandera para alternar entre la muestra local y el dataset completo (~8.3 GB)
# Nota: Para la entrega y ejecución final, cambiar a False.
USE_SAMPLE = True
DATA_PATH = SAMPLE_PATH if USE_SAMPLE else FULL_PATH
MAX_ROWS = None  # None indica que se leerá la totalidad del archivo seleccionado

print(f"Iniciando carga de datos desde: {DATA_PATH}")
modo = "Muestra de desarrollo" if USE_SAMPLE else "Dataset completo (~38.9M de transferencias)"
print(f"Modo de ejecución: {modo}")

# Inicialización y procesamiento del grafo
loader = EthereumGraphLoader()
loader.load_transfers(DATA_PATH, max_rows=MAX_ROWS, aggregate=True)
loader.load_node_labels(EXCHANGE_PATH, STABLE_PATH)

print("\nGrafo construido exitosamente:")
print(f" - Total de nodos (|V|): {loader.num_nodes:,}")
print(f" - Total de aristas agregadas (|E|): {loader.num_edges:,}")
print(f" - Nodos etiquetados (Exchanges/Stablecoins): {len(loader.node_meta):,}")

In [ ]:
# Análisis exploratorio y métricas generales del grafo

summary = loader.summary()
print("-" * 50)
print("Resumen de la red G = (V, E)")
print("-" * 50)
for k, v in summary.items():
    print(f"{k:<25}: {v}")
print("-" * 50)

# Cálculo y visualización de la distribución de grados
degrees = loader.degree_sequence()
out_degs = [d['out_degree'] for d in degrees.values()]
in_degs  = [d['in_degree']  for d in degrees.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma para el grado de salida (escala logarítmica)
axes[0].hist([d for d in out_degs if d > 0], bins=50, color='steelblue', edgecolor='white', log=True)
axes[0].set_title('Distribución Grado de Salida', fontsize=13)
axes[0].set_xlabel('Grado de Salida')
axes[0].set_ylabel('Frecuencia (escala log)')
axes[0].grid(alpha=0.3)

# Histograma para el grado de entrada (escala logarítmica)
axes[1].hist([d for d in in_degs if d > 0], bins=50, color='coral', edgecolor='white', log=True)
axes[1].set_title('Distribución Grado de Entrada', fontsize=13)
axes[1].set_xlabel('Grado de Entrada')
axes[1].set_ylabel('Frecuencia (escala log)')
axes[1].grid(alpha=0.3)

plt.suptitle('Red de Transferencias Ethereum: Distribución de Grados', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('assets/distribucion_grados.png', dpi=150, bbox_inches='tight')
plt.show()

print("Gráfica exportada exitosamente en: assets/distribucion_grados.png")

In [ ]:
# Visualización del subgrafo de nodos etiquetados
# Nota: NetworkX se utiliza en esta sección exclusivamente para la renderización del grafo.

if NX_AVAILABLE:
    # Construcción de un subgrafo enfocado en las entidades etiquetadas (exchanges/pools)
    labeled_set = set(loader.node_meta.keys())
    
    # Se filtran únicamente las aristas donde ambos extremos tienen etiquetas conocidas
    sub_edges = [(s, d, w) for s, d, w in loader.edges 
                 if s in labeled_set and d in labeled_set]
    
    if sub_edges:
        G_viz = nx.DiGraph()
        # Se limita la muestra para evitar saturación visual en el plot
        G_viz.add_weighted_edges_from(sub_edges[:2000])  
        
        pos = nx.spring_layout(G_viz, seed=42, k=0.5)
        node_labels = {n: loader.node_meta.get(n, {}).get('name', n[:8]) 
                       for n in G_viz.nodes()}
        
        plt.figure(figsize=(16, 10))
        nx.draw_networkx(G_viz, pos, labels=node_labels, 
                         node_size=300, font_size=7, arrows=True,
                         node_color='lightblue', edge_color='gray', alpha=0.8)
        
        plt.title('Subgrafo de Exchanges y Pools (Red Ethereum DeFi)', fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        plt.savefig('assets/subgrafo_exchanges.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print(f"Visualización generada: Subgrafo con {G_viz.number_of_nodes()} nodos y {G_viz.number_of_edges()} aristas.")
    else:
        print("No se encontró suficiente interconexión entre los nodos etiquetados para generar la visualización.")
else:
    print("NetworkX no está disponible. Para visualizar el subgrafo, instale la dependencia con: pip install networkx")

---
### Tarea 1: Caminos Mínimos (15%)
**Responsable:** Pablo Álvarez

#### 1.1 Contexto del Problema

En el análisis de la red de transferencias de tokens ERC-20, el cálculo de caminos de menor costo entre dos direcciones específicas (billeteras) es fundamental para rastrear rutas financieras, detectar posibles esquemas de ocultamiento de fondos o identificar ciclos de transacciones. 

El costo de una ruta puede definirse de forma topológica (número de intermediarios o saltos) o, en un modelo analítico más avanzado, utilizando el logaritmo negativo del volumen transferido, lo cual es útil para la detección de ciclos con pesos netos negativos.

#### 1.2 Formulación Matemática

Dado el grafo dirigido y ponderado $G = (V, E, w)$, el problema clásico de caminos mínimos busca la ruta de menor costo desde un nodo origen $s$ hacia cualquier otro nodo $v$:

$$\delta(s, v) = \min_{p: s \leadsto v} \sum_{e \in p} w(e) \quad \forall v \in V$$

Para la variante de detección de arbitraje o ciclos anómalos, se puede aplicar una transformación a los pesos originales:

$$w'(e) = -\log\left(\frac{\text{value}(e)}{\text{value}_{\max}}\right) \in \mathbb{R}$$

Bajo esta métrica, un ciclo de peso negativo en $w'$ representa matemáticamente una anomalía financiera u oportunidad de arbitraje.

#### 1.3 Análisis de Alternativas Algorítmicas

| Algoritmo | Complejidad Temporal | Soporte para Pesos Negativos | Veredicto |
|-----------|----------------------|------------------------------|-----------|
| **Dijkstra** | $O((V+E) \log V)$ | No soportado | **Prohibido (Restricción del curso)** |
| **Floyd-Warshall** | $O(V^3)$ | Soportado | Inviable por el orden de magnitud ($V \gg 10^3$) |
| **BFS** | $O(V+E)$ | No aplica ($w=1$) | Descartado (solo válido en grafos no ponderados) |
| **Bellman-Ford**| $O(V \cdot E)$ | Soportado (Detecta ciclos negativos) | **Seleccionado** |

**Justificación de la selección:** 
El algoritmo de Bellman-Ford cumple con todas las restricciones y necesidades del escenario planteado. Cumple la directriz del profesor al evitar el uso de Dijkstra y, a diferencia de este último, Bellman-Ford está diseñado estructuralmente para procesar grafos con pesos negativos y reportar explícitamente la existencia de ciclos de costo negativo, lo cual es fundamental para el análisis propuesto.

In [ ]:
# Construcción del subgrafo para el algoritmo de Bellman-Ford
# Debido a la complejidad de O(V*E), se extrae un subgrafo acotado que contiene 
# los nodos de mayor conectividad y las entidades conocidas (exchanges/pools).

import math

# Identificar el conjunto de nodos con etiquetas conocidas
labeled_set = set(loader.node_meta.keys())

# Calcular el grado de salida de cada nodo en el grafo original
degree_map = {v: len(loader.adj.get(v, [])) for v in loader.nodes}

# Seleccionar los 1000 nodos más activos como pivotes topológicos
top_nodes_bf = set(sorted(degree_map, key=degree_map.get, reverse=True)[:1000])

# Garantizar que todas las entidades etiquetadas formen parte del subgrafo
top_nodes_bf |= labeled_set

# Filtrar las adyacencias: conservar únicamente aristas donde ambos extremos 
# pertenezcan al conjunto de nodos seleccionados (top_nodes_bf)
sub_adj_bf = {
    src: [(dst, w) for dst, w in neighbors if dst in top_nodes_bf]
    for src, neighbors in loader.adj.items()
    if src in top_nodes_bf
}

total_aristas = sum(len(v) for v in sub_adj_bf.values())

print("Subgrafo para Bellman-Ford generado con éxito:")
print(f" - Nodos seleccionados: {len(top_nodes_bf):,}")
print(f" - Aristas internas preservadas: {total_aristas:,}")

In [ ]:
# Ejecución del algoritmo de Bellman-Ford y análisis de rutas
# Se busca identificar los caminos mínimos y validar la existencia de ciclos anómalos.

# Identificación de un nodo de origen (source) relevante, preferiblemente un DEX (Decentralized Exchange)
source_node = None
for addr, meta in loader.node_meta.items():
    if meta.get('type') == 'dex' and addr in sub_adj_bf:
        source_node = addr
        source_name = meta.get('name', addr[:12])
        break

# Alternativa: Si no se encuentra un DEX, se selecciona el nodo de mayor grado de salida
if source_node is None:
    source_node = max(sub_adj_bf, key=lambda n: len(sub_adj_bf.get(n, [])))
    source_name = f"Nodo de alto grado ({source_node[:12]}...)"

print(f"Nodo fuente seleccionado: {source_name} ({source_node[:20]}...)")
print("Ejecutando algoritmo de Bellman-Ford...")

try:
    dist, prev = bellman_ford(sub_adj_bf, source_node)
    
    # Filtrar únicamente los nodos que tienen un camino válido desde el origen
    reachable = {v: d for v, d in dist.items() if d < float('inf')}
    
    print("\nEjecución completada exitosamente.")
    print(f" - Nodos alcanzables desde {source_name}: {len(reachable):,}")
    print(f" - Nodos inalcanzables: {len(dist) - len(reachable):,}")
    
    # Extracción de los 5 nodos más distantes según el peso calculado de la ruta
    top_distant = sorted(reachable.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 nodos más distantes (mayor costo de ruta):")
    for addr, cost in top_distant:
        name = loader.node_meta.get(addr, {}).get('name', addr[:16] + '...')
        print(f"   {name:<35} costo = {cost:.4e}")

except ValueError as e:
    # Captura de la excepción lanzada por la detección de un ciclo de costo negativo
    print(f"\nAlerta algorítmica: Ciclo de peso negativo detectado por Bellman-Ford.")
    print(f"Detalle: {e}")
    print("Interpretación: Esto indica una posible oportunidad de arbitraje o ciclo de lavado de fondos dentro del subgrafo.")

In [ ]:
# Reconstrucción de la ruta óptima entre dos entidades (Exchanges/DEX)
# Se utiliza el arreglo de predecesores generado por Bellman-Ford para trazar el camino exacto.

# Búsqueda de un nodo destino válido (etiquetado) que sea alcanzable desde el origen
target_node = None
for addr, meta in loader.node_meta.items():
    if addr != source_node and meta.get('type') in ('dex', 'exchange') and addr in dist:
        if dist[addr] < float('inf'):
            target_node = addr
            target_name = meta.get('name', addr[:12])
            break

if target_node:
    path = reconstruct_path(prev, source_node, target_node)
    
    print(f"Camino mínimo reconstruido: {source_name} -> {target_name}")
    print(f" - Costo total acumulado: {dist[target_node]:.4e}")
    print(f" - Longitud topológica (saltos): {len(path) - 1}")
    
    print("\nSecuencia de la ruta:")
    for i, node in enumerate(path):
        # Se obtiene el nombre de la entidad si existe, o se trunca la dirección
        name = loader.node_meta.get(node, {}).get('name', node[:20] + '...')
        connector = " -> " if i < len(path) - 1 else ""
        print(f"   [{i}] {name}{connector}")
else:
    print("No se encontro un nodo destino con etiqueta conocida que sea alcanzable desde el origen")
    print("Sugerencia metodológica: Ampliar la muestra del subgrafo o definir manualmente los nodos")

---
### Tarea 2: Centralidad e Influencia (15%)
**Responsable:** Jhon Arciniegas

#### 2.1 Contexto del Problema

El ecosistema de finanzas descentralizadas (DeFi) opera a través de contratos inteligentes que actúan como intermediarios de liquidez. La identificación de nodos sistémicamente importantes dentro de la topología de la red permite:
* Detectar protocolos de alto riesgo topológico (puntos únicos de fallo o *single points of failure*).
* Identificar direcciones con alto poder de influencia (grandes tenedores de liquidez o bots de arbitraje).
* Evaluar el nivel de descentralización real de la red en contraste con la descentralización teórica.

#### 2.2 Formulación Matemática

Sobre el modelo del grafo $G = (V, E, w)$, se establecen las siguientes métricas de análisis:

**Centralidad de Grado Ponderado:**
$$C_D^w(v) = \frac{\sum_{u \in N(v)} w(v,u)}{(|V|-1) \cdot w_{\max}} \in [0, 1]$$

**PageRank (Poder de atracción de flujo):**
$$PR(v) = \frac{1-d}{|V|} + d \cdot \sum_{u \in \text{in}(v)} \frac{PR(u)}{\deg^+(u)}$$

**Centralidad de Intermediación (Influencia sobre el flujo de rutas):**
$$C_B(v) = \sum_{s \neq v \neq t} \frac{\sigma(s,t|v)}{\sigma(s,t)}$$

#### 2.3 Análisis de Alternativas Algorítmicas

| Métrica | Algoritmo Base | Complejidad Temporal | Veredicto |
|---------|----------------|----------------------|-----------|
| Closeness Centrality | Dijkstra | $O((V+E) \log V)$ por nodo | **Prohibido (Restricción del curso)** |
| Degree Centrality | Conteo directo | $O(V+E)$ | Implementado como línea base |
| PageRank | Iteración de Potencias (*Power Iteration*) | $O(\text{iter} \cdot E)$ | **Seleccionado** |
| Betweenness | Algoritmo de Brandes (BFS) | $O(V \cdot (V+E))$ | **Seleccionado** (Aproximación por muestreo) |
| HITS (Hubs/Authorities)| Iterativo | $O(\text{iter} \cdot E)$ | Descartado (Menor interpretabilidad transaccional) |

In [ ]:
# Cálculo y análisis de la Centralidad de Grado (Degree Centrality)
# Se evalúan los grados de salida (emisión) y entrada (recepción) para identificar 
# los nodos con mayor volumen de conexiones directas en la red.

print("Calculando métricas de centralidad de grado...")
deg_cent_out = degree_centrality(loader.adj, loader.nodes, mode='out')
deg_cent_in  = degree_centrality(loader.adj, loader.nodes, mode='in')

print("\nTop 10 - Nodos emisores (Mayor grado de salida):")
top_out = top_k_nodes(deg_cent_out, k=10, label_map=loader.node_meta)
for i, (addr, score, name) in enumerate(top_out, 1):
    print(f"  {i:2}. {name:<35} C_D(out) = {score:.6f}")

print("\nTop 10 - Nodos receptores (Mayor grado de entrada):")
top_in = top_k_nodes(deg_cent_in, k=10, label_map=loader.node_meta)
for i, (addr, score, name) in enumerate(top_in, 1):
    print(f"  {i:2}. {name:<35} C_D(in)  = {score:.6f}")

In [ ]:
# Cálculo de centralidad mediante PageRank (Implementación propia)
# Se evalúa el poder de atracción de flujo de liquidez de los nodos en la red.
# Por restricciones de tiempo de ejecución en Python puro, la convergencia 
# se calcula sobre el subgrafo de mayor conectividad e interés.

print("Iniciando cálculo de PageRank (Iteración de potencias sobre subgrafo principal)...")

# Se reutiliza el conjunto de nodos de alta relevancia (top-1000 + etiquetados)
sub_nodes = top_nodes_bf  
sub_adj_pr = {
    src: [(dst, w) for dst, w in neighbors if dst in sub_nodes]
    for src, neighbors in loader.adj.items()
    if src in sub_nodes
}

# Ejecución del algoritmo con un factor de amortiguación (damping factor) estándar de 0.85
pr_scores = pagerank(sub_adj_pr, sub_nodes, damping=0.85, max_iter=100)

print("\nTop 10 - Nodos con mayor influencia estructural (PageRank):")
top_pr = top_k_nodes(pr_scores, k=10, label_map=loader.node_meta)
for i, (addr, score, name) in enumerate(top_pr, 1):
    print(f"  {i:2}. {name:<35} PR = {score:.8f}")

In [ ]:
# Cálculo de la Centralidad de Intermediación (Betweenness Centrality)
# Se utiliza una aproximación del algoritmo de Brandes mediante muestreo aleatorio.
# En estricto cumplimiento de las restricciones del proyecto, los caminos mínimos 
# se calculan topológicamente usando BFS (saltos/hops), garantizando la exclusión de Dijkstra.

print("Calculando Centralidad de Intermediación (Aproximación de Brandes con 100 fuentes)...")

# Ejecución del algoritmo sobre el subgrafo con semilla fija para reproducibilidad
btw_scores = betweenness_centrality_approx(
    sub_adj_pr, sub_nodes, sample_size=100, seed=42
)

print("\nTop 10 - Nodos puente o intermediarios (Mayor Betweenness):")
print("Indicador de las direcciones con mayor control sobre las rutas de transferencia.")
top_btw = top_k_nodes(btw_scores, k=10, label_map=loader.node_meta)
for i, (addr, score, name) in enumerate(top_btw, 1):
    print(f"  {i:2}. {name:<35} C_B = {score:.8f}")

In [ ]:
# Visualización comparativa de las métricas de centralidad
# Se grafican los 8 nodos principales para cada métrica (Grado, PageRank e Intermediación),
# permitiendo contrastar los diferentes tipos de influencia estructural en la red.

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Definición de las métricas a visualizar: (scores, título, color, top_k calculado previamente)
metrics = [
    (deg_cent_out, 'Centralidad de Grado\n(Nodos emisores)', 'steelblue', top_out),
    (pr_scores,    'PageRank\n(Atracción de flujo)', 'darkorange', top_pr),
    (btw_scores,   'Centralidad de Intermediación\n(Control de rutas)', 'forestgreen', top_btw),
]

for ax, (scores, title, color, top) in zip(axes, metrics):
    # Formateo de nombres para evitar superposición de texto en el eje Y
    names = [t[2][:18] if t[2] != 'unknown' else t[0][:12] + '...' for t in top[:8]]
    vals  = [t[1] for t in top[:8]]
    
    ax.barh(range(len(names)), vals, color=color, alpha=0.8, edgecolor='white')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Puntuación (Score)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()  # El nodo de mayor puntaje aparece en la parte superior
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Comparativa de Nodos Sistémicamente Importantes - Red Ethereum DeFi', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('assets/centralidad_comparativa.png', dpi=150, bbox_inches='tight')
plt.show()

print("Gráfica comparativa exportada exitosamente en: assets/centralidad_comparativa.png")

---
### Tarea 3: Detección de Comunidades (15%)
**Responsable:** Emerson Ballesteros (Líder)

#### 3.1 Contexto del Problema

La red de finanzas descentralizadas (DeFi) en Ethereum presenta una topología marcadamente heterogénea. Existen ecosistemas de protocolos y entidades que interactúan de forma densa entre sí (por ejemplo, los ecosistemas de Uniswap, Curve/Convex o Aave/Compound). La detección algorítmica de estas comunidades permite:
* Visualizar e interpretar la macroestructura modular del ecosistema financiero.
* Detectar patrones de coordinación anómala o grupos de entidades altamente correlacionadas (ej. redes de arbitraje automatizado o *bot swarms*).
* Evaluar el grado de descentralización real del ecosistema frente a la formación de clústeres dominantes.

#### 3.2 Formulación Matemática

El problema se aborda mediante la optimización de la **Modularidad de Newman-Girvan** en el grafo $G = (V, E)$. La modularidad mide la densidad de las aristas dentro de las comunidades frente a la densidad esperada en un grafo aleatorio equivalente:

$$Q = \frac{1}{2m} \sum_{i,j} \left[A_{ij} - \frac{k_i k_j}{2m}\right] \delta(c_i, c_j)$$

Donde:
* $m = |E|$ es el número total de aristas.
* $A_{ij}$ es el peso de la arista entre $i$ y $j$.
* $k_i, k_j$ son los grados de los nodos $i$ y $j$.
* $\delta(c_i, c_j)$ es la función delta de Kronecker (vale 1 si $i$ y $j$ pertenecen a la misma comunidad $c$, y 0 en caso contrario).

**Objetivo de optimización:** Maximizar $Q \in [-0.5, 1.0]$. En la práctica de análisis de redes, valores de $Q > 0.3$ sugieren la existencia de una estructura comunitaria significativa.

#### 3.3 Análisis de Alternativas Algorítmicas

| Algoritmo | Complejidad Temporal | Escala Soportada | Veredicto |
|-----------|----------------------|------------------|-----------|
| Girvan-Newman | $O(V \cdot E^2)$ | Pequeña | Inviable computacionalmente para $|E| > 10^5$ |
| Spectral Clustering | $O(V^3)$ | Mediana | Descartado (Alta demanda en operaciones de álgebra lineal densa) |
| Label Propagation | $O(V+E)$ | Grande | Descartado (Alta velocidad, pero convergencia inestable) |
| **Louvain** | $O(V \log V)$ empírico| Muy grande | **Seleccionado** |
| Variantes con Dijkstra| — | — | **Prohibido (Restricción del curso)** |

**Justificación de la selección:**
El método de Louvain ofrece el mejor balance entre precisión en la maximización de la modularidad $Q$ y eficiencia computacional. Su aproximación heurística jerárquica es ideal para redes del volumen transaccional de Ethereum, y su lógica de agregación local no depende de algoritmos de caminos mínimos como Dijkstra.

In [ ]:
# Detección de comunidades mediante el algoritmo de Louvain (Implementación propia)
# Nota metodológica: El método estándar de Louvain opera sobre topologías no dirigidas,
# por lo cual se requiere simetrizar las aristas del subgrafo original antes de la ejecución.

print("Preparando transformación a grafo no dirigido...")

# Se extrae la red de adyacencia enfocada en los nodos de mayor relevancia
sub_adj_comm = {
    src: [(dst, w) for dst, w in neighbors if dst in sub_nodes]
    for src, neighbors in loader.adj.items()
    if src in sub_nodes
}

# Simetrización matemática del grafo
adj_sym = symmetrize_graph(sub_adj_comm)
print(f" - Nodos procesados en la matriz simétrica: {len(adj_sym):,}")

print("Ejecutando algoritmo de Louvain (Fases de optimización local y agregación)...")
# Se fija la semilla (seed=42) para garantizar la reproducibilidad académica del clustering
community_map = louvain_communities(adj_sym, resolution=1.0, max_passes=5, seed=42)

# Cálculo de la Modularidad de Newman-Girvan (Q) para validar la calidad del particionamiento
Q = compute_modularity(adj_sym, community_map)
num_communities = len(set(community_map.values()))

print("\nDetección completada exitosamente.")
print(f" - Número de comunidades detectadas: {num_communities}")

# Interpretación técnica de la modularidad
calidad_estructura = "Estructura comunitaria significativa" if Q > 0.3 else "Estructura comunitaria débil/difusa"
print(f" - Modularidad alcanzada (Q): {Q:.4f} -> {calidad_estructura}")

In [ ]:
# Análisis estadístico y descriptivo de las comunidades detectadas
# Se genera un resumen de las agrupaciones principales para identificar 
# los ecosistemas dominantes dentro de la red (ej. protocolos DeFi o exchanges centralizados).

comm_summary = community_summary(community_map, loader.node_meta)

print("Top 10 comunidades por volumen de nodos:")
print(f"{'Rank':<5} {'ID':<8} {'Tamaño':<10} {'Nodos Etiquetados':<20} {'Protocolos Conocidos'}")
print("-" * 80)
for rank, (comm_id, info) in enumerate(list(comm_summary.items())[:10], 1):
    names = ', '.join(info['names'][:3])
    if len(info['names']) > 3:
        names += f" ... (+{len(info['names'])-3})"
    print(f"{rank:<5} {comm_id:<8} {info['size']:<10} {len(info['labeled']):<20} {names or '(Sin etiquetas conocidas)'}")

# Visualización de la distribución de tamaños para evaluar la concentración de la red
sizes = [info['size'] for info in comm_summary.values()]
plt.figure(figsize=(10, 5))

# Se utiliza escala logarítmica dado el sesgo topológico esperado en redes de libre escala
plt.hist(sizes, bins=min(30, len(sizes)), color='mediumpurple', edgecolor='white', log=True)
plt.xlabel('Tamaño de la comunidad (Número de nodos)')
plt.ylabel('Frecuencia (Escala logarítmica)')
plt.title(f'Distribución de Tamaños de Comunidades (Q = {Q:.4f})', fontsize=13)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('assets/comunidades_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nHistograma de distribución exportado exitosamente en: assets/comunidades_distribucion.png")

---
### Tarea 4: Árbol de Expansión Mínima (15%)
**Responsabilidad:** Grupal (Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros)

#### 4.1 Contexto del Problema

El cálculo del Árbol de Expansión Mínima (MST) sobre el grafo de transferencias permite identificar la red de conectividad topológica mínima que mantiene cohesionado al ecosistema DeFi. En términos de análisis de riesgo estructural, este enfoque responde a la siguiente interrogante: 

*¿Cuál es el conjunto de rutas de menor volumen que, en caso de interrupción, fragmentaría el ecosistema en componentes desconectados?*

Bajo esta perspectiva, las aristas del MST representan los puentes críticos de liquidez. Análogamente, el cálculo de un Árbol de Expansión Máxima (MaxST) sobre el mismo grafo permite abstraer la columna vertebral del ecosistema (las rutas de mayor volumen y recurrencia).

#### 4.2 Formulación Matemática

Para un grafo conexo, no dirigido y ponderado $G = (V, E, w)$, el MST se define como el subgrafo $T^*$ que minimiza la suma de sus pesos estructurales:

$$T^* = \arg\min_{T \in \text{span}(G)} \sum_{e \in T} w(e)$$

**Propiedades formales a garantizar en la implementación:**
* Conservación de vértices y aristas: $|V(T^*)| = |V(G)|$ y $|E(T^*)| = |V(G)| - 1$.
* $T^*$ es un subgrafo conexo y estrictamente acíclico.
* **Propiedad del Corte:** Para todo corte $(S, V\setminus S)$, la arista de menor peso que cruza dicho corte pertenece obligatoriamente a $T^*$.
* **Propiedad del Ciclo:** Para todo ciclo $C$ en el grafo original, la arista de mayor peso de $C$ no puede pertenecer a $T^*$.

#### 4.3 Análisis de Alternativas Algorítmicas

| Algoritmo | Complejidad Temporal | Complejidad de Implementación | Veredicto |
|-----------|----------------------|-------------------------------|-----------|
| Dijkstra (Variante) | — | — | **Prohibido (Restricción del curso)** |
| Prim (con Min-Heap) | $O((V+E) \log V)$ | Moderada | Válido computacionalmente |
| Borůvka | $O(E \log V)$ | Alta | Válido, pero propenso a errores en Python puro |
| **Kruskal (con DSU)** | $O(E \log E)$ | Modular y verificable | **Seleccionado** |

**Justificación de la selección:**
El algoritmo de Kruskal, estructurado mediante un modelo de Conjuntos Disjuntos (Union-Find con heurísticas de compresión de caminos y unión por rango), resulta ser la aproximación metodológica más robusta para evaluar los invariantes de los ciclos. Asimismo, la cota de complejidad logarítmica impuesta por el ordenamiento inicial de aristas lo hace idóneo para analizar redes dispersas de la escala que maneja Ethereum.

In [ ]:
# Cálculo del Árbol de Expansión Mínima (MST) y Máxima (MaxST)
# Se utiliza el algoritmo de Kruskal apoyado en una estructura de Conjuntos Disjuntos (Union-Find).

print("Preparando y simetrizando aristas para el algoritmo de Kruskal...")

# Transformación del subgrafo dirigido a no dirigido (agregando pesos topológicos)
directed_sub_edges = [
    (src, dst, w)
    for src, neighbors in sub_adj_comm.items()
    for dst, w in neighbors
]
undirected_edges = symmetrize_edges(directed_sub_edges)

print(f" - Aristas dirigidas procesadas: {len(directed_sub_edges):,}")
print(f" - Aristas no dirigidas resultantes: {len(undirected_edges):,}")

# Ejecución de Kruskal para obtener el Minimum Spanning Tree (MST)
# Representa los puentes críticos de conectividad con el menor volumen de transferencia.
print("\nEjecutando algoritmo de Kruskal para Árbol de Expansión Mínima...")
mst_edges, mst_total_weight = kruskal_mst(sub_nodes, undirected_edges, maximize=False)

print("Validación topológica del MST completada.")
print(f" - Aristas integradas al árbol: {len(mst_edges):,} (Valor teórico esperado: {len(sub_nodes)-1:,})")
print(f" - Peso total de la red mínima: {mst_total_weight:.4e}")

# Ejecución de Kruskal para obtener el Maximum Spanning Tree (MaxST)
# Representa la columna vertebral (backbone) de mayor flujo de liquidez en el ecosistema.
print("\nEjecutando algoritmo de Kruskal para Árbol de Expansión Máxima (Backbone)...")
maxst_edges, maxst_total_weight = kruskal_mst(sub_nodes, undirected_edges, maximize=True)

print(f" - Peso total de la estructura de máximo flujo: {maxst_total_weight:.4e}")

In [ ]:
# Análisis estadístico y estructural del Árbol de Expansión Mínima
# Se extraen métricas clave para interpretar la vulnerabilidad y distribución 
# de los puentes que mantienen cohesionada la topología de la red.

analytics = mst_analytics(mst_edges, loader.node_meta)

print("Métricas estructurales del Árbol de Expansión Mínima (MST):")
print(f" - Número de aristas (conexiones vitales): {analytics['num_edges']:,}")
print(f" - Volumen total retenido en el árbol:     {analytics['total_weight']:.4e}")
print(f" - Volumen promedio por conexión:          {analytics['avg_weight']:.4e}")
print(f" - Conexión de máximo volumen (cuello de botella): {analytics['max_weight']:.4e}")
print(f" - Conexión de mínimo volumen:             {analytics['min_weight']:.4e}")

print("\nTop 10 - Enlaces críticos del MST (Mayor volumen transferido):")
print("Estas rutas representan las dependencias de liquidez más fuertes dentro de la estructura de conectividad mínima.")
print(f"{'Origen (From)':<30} {'Destino (To)':<30} {'Volumen'}")
print("-" * 80)
for edge in analytics.get('top_weight_edges', []):
    print(f"  {edge['from']:<30} {edge['to']:<30} {edge['weight']:.4e}")

In [ ]:
# Visualización topológica del Árbol de Expansión Mínima (MST)
# Nota: El uso de NetworkX en esta sección se limita estrictamente a la renderización 
# visual del árbol previamente calculado, respetando las restricciones del proyecto.

if NX_AVAILABLE:
    # Inicialización del grafo de renderizado
    T = nx.Graph()
    
    # Filtrado de aristas: se conservan aquellas donde al menos un extremo tiene etiqueta conocida
    mst_labeled = [(s, d, w) for s, d, w in mst_edges 
                   if s in loader.node_meta or d in loader.node_meta]
    
    # Se limita el número de aristas para mantener la legibilidad de la gráfica
    T.add_weighted_edges_from(mst_labeled[:500])  
    
    if T.number_of_nodes() > 0:
        # Algoritmo de disposición (layout) basado en fuerzas (Kamada-Kawai)
        pos = nx.kamada_kawai_layout(T)
        
        # Asignación de paleta de colores según la clasificación de la entidad
        colors = []
        for n in T.nodes():
            meta = loader.node_meta.get(n, {})
            ntype = meta.get('type', '')
            if 'uniswap' in ntype: 
                colors.append('royalblue')
            elif 'dex' in ntype: 
                colors.append('darkorange')
            elif ntype: 
                colors.append('forestgreen')
            else: 
                colors.append('lightgray')
        
        # Etiquetado y escalado de grosores de aristas según el volumen transferido
        labels = {n: loader.node_meta.get(n, {}).get('name', n[:8]) 
                  for n in T.nodes() if n in loader.node_meta}
        weights = [T[u][v]['weight'] for u, v in T.edges()]
        max_w = max(weights) if weights else 1
        edge_widths = [1 + 4 * (w / max_w) for w in weights]
        
        plt.figure(figsize=(16, 12))
        
        # Renderizado de componentes
        nx.draw_networkx_nodes(T, pos, node_color=colors, node_size=200, alpha=0.9)
        nx.draw_networkx_edges(T, pos, width=edge_widths, edge_color='steelblue', alpha=0.5)
        nx.draw_networkx_labels(T, pos, labels=labels, font_size=7, font_weight='bold')
        
        plt.title(f'Árbol de Expansión Mínima (Red Ethereum DeFi) - {T.number_of_nodes()} nodos, {T.number_of_edges()} aristas', fontsize=13)
        plt.axis('off')
        plt.tight_layout()
        plt.savefig('assets/mst_defi.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("Visualización del MST exportada exitosamente en: assets/mst_defi.png")
    else:
        print("No se encontraron suficientes nodos etiquetados en el MST para generar la visualización.")
else:
    print("NetworkX no está disponible. Para renderizar el MST, instale la dependencia con: pip install networkx")

---
### Resultados Esperados y Criterios de Evaluación

#### 1. Objetivos de Análisis por Tarea

| Tarea | Objetivo Analítico |
|-------|--------------------|
| **T1 – Caminos Mínimos (Bellman-Ford)** | Aislar topológicamente las rutas de transferencia entre exchanges y evaluar la viabilidad matemática de detectar ciclos de arbitraje mediante la transformación logarítmica de volúmenes. |
| **T2 – Centralidad e Influencia** | Identificar los *hubs* sistémicos y concentradores primarios de liquidez en la red (ej. contratos inteligentes dominantes) a través del cálculo de PageRank y Centralidad de Grado. |
| **T3 – Estructura Comunitaria (Louvain)** | Revelar la división macro-modular de la red maximizando la métrica de modularidad $Q$, con el fin de identificar ecosistemas de alta cohesión interna. |
| **T4 – Árboles de Expansión (Kruskal)** | Construir el MST para identificar los puentes críticos de liquidez (puntos de riesgo sistémico) y el MaxST para mapear la columna vertebral de los flujos del mercado. |

#### 2. Criterios Técnicos de Aceptación (A cumplir durante el desarrollo)
Para garantizar el rigor académico exigido en el proyecto, el equipo se compromete a respetar las siguientes restricciones en la implementación:
* **Exclusión de Dijkstra:** El algoritmo de Dijkstra debe ser omitido en la totalidad de las rutinas de cálculo topológico.
* **Implementación Nativa:** Toda la lógica algorítmica y el manejo de estructuras de datos (grafos, árboles, conjuntos disjuntos) se desarrollará utilizando exclusivamente Python puro.
* **Uso Restringido de Librerías:** La dependencia `networkx` se limitará estricta y exclusivamente a las etapas de renderizado visual y graficación, sin intervenir en la lógica de grafos.

#### 3. Proyecciones y Posibles Extensiones
Una vez completada la implementación base, se contemplan las siguientes vías de expansión:
* **Escalabilidad Algorítmica:** Extender la ejecución de los algoritmos al dataset completo (~38.9 millones de aristas) implementando técnicas de procesamiento por lotes.
* **Resolución Multi-escala:** Aplicar el método de Louvain iterando sobre el parámetro de resolución para descubrir sub-comunidades anidadas.
* **Correlación de Riesgo:** Analizar cómo los nodos identificados en el MST y por las métricas de centralidad podrían afectar la red ante caídas abruptas de liquidez.

In [ ]:
# Reporte de estado actual del proyecto
# Resumen de métricas estructurales y estado de implementación de los módulos

print("-" * 75)
print("ESTADO DEL PROYECTO - TEORÍA DE GRAFOS (UIS)")
print("-" * 75)
print("Dataset cargado: Ethereum Token Transfers (ERC-20)")
print(f" - Total de nodos (|V|):     {loader.num_nodes:>14,}")
print(f" - Total de aristas (|E|):   {loader.num_edges:>14,}")
print(f" - Densidad de la red (δ):   {loader.density():>14.6e}")

print("\nEstado de módulos y responsabilidades (Archivos .py en src/):")
print(" [T1] Caminos Mínimos (Bellman-Ford)       -> Pablo Álvarez       [PENDIENTE]")
print(" [T2] Centralidad (PageRank/Betweenness)   -> Jhon Arciniegas     [PENDIENTE]")
print(" [T3] Comunidades (Louvain)                -> Emerson Ballesteros [PENDIENTE]")
print(" [T4] Árboles de Expansión (Kruskal MST)   -> Asignación Grupal   [PENDIENTE]")

print("\nValidación de restricciones técnicas (Objetivos):")
print(" - Política de exclusión de Dijkstra:    ESTRICTAMENTE REQUERIDO")
print(" - Implementación en Python puro:        ESTRICTAMENTE REQUERIDO")
print("-" * 75)